In [1]:
# churn_pipeline_ensemble.py
# Requirements:
# pip install scikit-learn xgboost imbalanced-learn pandas numpy matplotlib

import warnings
warnings.filterwarnings("ignore", category=UserWarning)

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             classification_report)
from scipy.stats import randint, uniform
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
import time

RANDOM_STATE = 42

# -------------------------
# 1) Load data (edit path & target)
# -------------------------
# The dataset must have a binary target column named 'Churn' (0/1 or 'No'/'Yes').
# Change 'churn.csv' to your file and adjust categorical/numeric column lists.
df = pd.read_csv("Customer_Churn.csv")  # <-- change path to your CSV

# Example: if 'Churn' is 'Yes'/'No', convert to 1/0
if df['Churn'].dtype == object:
    df['Churn'] = df['Churn'].map({'No': 0, 'Yes': 1})

target_col = 'Churn'
y = df[target_col]
X = df.drop(columns=[target_col])

# -------------------------
# 2) Identify categorical & numeric columns (edit if necessary)
# -------------------------
# A simple heuristic; you can manually list columns if you prefer.
cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()

print("Categorical columns:", cat_cols)
print("Numeric columns:", num_cols)

# -------------------------
# 3) Train/test split
# -------------------------
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

# We'll keep a small validation split for XGBoost early stopping during hyperparam search
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.15, stratify=y_train_full, random_state=RANDOM_STATE
)

# -------------------------
# 4) Preprocessing Pipelines
# -------------------------
# Numeric transformer
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Categorical transformer
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, num_cols),
    ('cat', categorical_transformer, cat_cols)
])

# -------------------------
# 5) Models and param distributions for RandomizedSearchCV
# -------------------------
# 5.1 Logistic Regression
pipe_lr = Pipeline(steps=[
    ('preproc', preprocessor),
    # If you want to use SMOTE, use the ImbPipeline below instead (see example)
    ('clf', LogisticRegression(random_state=RANDOM_STATE, max_iter=1000))
])

param_dist_lr = {
    'clf__penalty': ['l2'],             # keep simple; change if you want l1 with solver='saga'
    'clf__C': uniform(0.01, 10),
    'clf__solver': ['lbfgs', 'saga']
}

# 5.2 Random Forest
pipe_rf = Pipeline(steps=[
    ('preproc', preprocessor),
    ('clf', RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1))
])

param_dist_rf = {
    'clf__n_estimators': randint(50, 400),
    'clf__max_depth': randint(3, 30),
    'clf__min_samples_split': randint(2, 10),
    'clf__min_samples_leaf': randint(1, 8),
    'clf__bootstrap': [True, False]
}

# 5.3 XGBoost
# Use XGBClassifier (scikit-learn wrapper)
pipe_xgb = Pipeline(steps=[
    ('preproc', preprocessor),
    ('clf', XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=RANDOM_STATE))
])

param_dist_xgb = {
    'clf__n_estimators': randint(50, 1000),
    'clf__learning_rate': uniform(0.01, 0.5),
    'clf__max_depth': randint(3, 12),
    'clf__subsample': uniform(0.5, 0.5),
    'clf__colsample_bytree': uniform(0.3, 0.7),
    'clf__reg_alpha': uniform(0, 1),
    'clf__reg_lambda': uniform(0, 2)
}

# -------------------------
# 6) Helper: run RandomizedSearchCV
# -------------------------
def randomized_search(pipe, param_dist, X_tr, y_tr, model_name,
                      n_iter=30, scoring='roc_auc', n_jobs=-1, random_state=RANDOM_STATE,
                      early_stop_val=None):
    """
    Run RandomizedSearchCV. If early_stop_val is provided (X_val, y_val),
    it will be passed as fit_params for models that accept eval_set (like XGB).
    """
    print(f"\n--- Starting RandomizedSearchCV for {model_name} ---")
    rs = RandomizedSearchCV(
        pipe,
        param_distributions=param_dist,
        n_iter=n_iter,
        scoring=scoring,
        n_jobs=n_jobs,
        cv=StratifiedKFold(n_splits=4, shuffle=True, random_state=random_state),
        random_state=random_state,
        verbose=1,
        refit=True
    )

    fit_kwargs = {}
    # if model is XGB, pass eval_set and early_stopping_rounds
    if early_stop_val is not None and 'XGB' in model_name.upper():
        Xv, yv = early_stop_val
        # RandomizedSearchCV will call .fit(X_tr, y_tr, **fit_kwargs) internally
        fit_kwargs = {'clf__early_stopping_rounds': 50,
                      'clf__eval_set': [(Xv, yv)],
                      'clf__verbose': False}

    start = time.time()
    rs.fit(X_tr, y_tr, **fit_kwargs)
    elapsed = time.time() - start
    print(f"RandomizedSearchCV done for {model_name} in {elapsed:.1f}s")
    print("Best params:", rs.best_params_)
    print("Best CV score:", rs.best_score_)
    return rs

# -------------------------
# 7) Option: use SMOTE inside pipeline (if dataset is imbalanced)
# Uncomment and swap `pipe_*` variables above if you want SMOTE applied automatically.
# Example:
# pipe_lr = ImbPipeline(steps=[('preproc', preprocessor), ('smote', SMOTE(random_state=RANDOM_STATE)), ('clf', LogisticRegression(...))])
# -------------------------

# -------------------------
# 8) Run hyperparameter tuning for each model
# -------------------------
# NOTE: tune iterations (n_iter) for RandomizedSearchCV depending on compute budget
rs_lr = randomized_search(pipe_lr, param_dist_lr, X_train, y_train, model_name='LogisticRegression', n_iter=30)
rs_rf = randomized_search(pipe_rf, param_dist_rf, X_train, y_train, model_name='RandomForest', n_iter=40)
# For XGBoost, pass validation set for early stopping
rs_xgb = randomized_search(pipe_xgb, param_dist_xgb, X_train, y_train, model_name='XGBoost', n_iter=50, early_stop_val=(X_val, y_val))

# -------------------------
# 9) Evaluate best models on test set
# -------------------------
best_lr = rs_lr.best_estimator_
best_rf = rs_rf.best_estimator_
best_xgb = rs_xgb.best_estimator_

models = {
    'LogisticRegression': best_lr,
    'RandomForest': best_rf,
    'XGBoost': best_xgb
}

def evaluate_model(name, model, X_t, y_t):
    y_pred = model.predict(X_t)
    y_proba = model.predict_proba(X_t)[:, 1] if hasattr(model, "predict_proba") else None

    acc = accuracy_score(y_t, y_pred)
    prec = precision_score(y_t, y_pred, zero_division=0)
    rec = recall_score(y_t, y_pred, zero_division=0)
    f1 = f1_score(y_t, y_pred, zero_division=0)
    roc = roc_auc_score(y_t, y_proba) if y_proba is not None else np.nan
    cm = confusion_matrix(y_t, y_pred)
    print(f"\n---- {name} Evaluation ----")
    print(f"Accuracy: {acc:.4f} | Precision: {prec:.4f} | Recall: {rec:.4f} | F1: {f1:.4f} | ROC-AUC: {roc:.4f}")
    print("Confusion matrix:\n", cm)
    print("Classification report:\n", classification_report(y_t, y_pred, zero_division=0))
    return {'accuracy': acc, 'precision': prec, 'recall': rec, 'f1': f1, 'roc_auc': roc}

results = {}
for name, model in models.items():
    results[name] = evaluate_model(name, model, X_test, y_test)

# -------------------------
# 10) Ensemble: VotingClassifier (hard or soft voting)
# Use the best estimators from tuning. We use soft voting if all models support predict_proba.
# -------------------------
voting = VotingClassifier(
    estimators=[
        ('lr', best_lr),
        ('rf', best_rf),
        ('xgb', best_xgb)
    ],
    voting='soft',  # 'soft' uses predict_proba and often performs better; 'hard' uses majority vote
    n_jobs=-1
)

print("\nFitting VotingClassifier (ensemble) on full train set...")
voting.fit(X_train_full, y_train_full)  # fit on entire train data for final ensemble

results['VotingEnsemble'] = evaluate_model('VotingEnsemble', voting, X_test, y_test)

# -------------------------
# 11) Optionally: pick the single best model by chosen metric (e.g., accuracy or roc_auc)
# -------------------------
best_by_acc = max(results.items(), key=lambda kv: kv[1]['accuracy'])
best_by_roc = max(results.items(), key=lambda kv: kv[1]['roc_auc'])

print(f"\nBest by Accuracy: {best_by_acc[0]} -> {best_by_acc[1]}")
print(f"Best by ROC-AUC: {best_by_roc[0]} -> {best_by_roc[1]}")

# -------------------------
# 12) Save models if desired (joblib)
# -------------------------
# from joblib import dump
# dump(best_xgb, "best_xgb.joblib")
# dump(voting, "voting_ensemble.joblib")

# -------------------------
# 13) Notes / How to customize
# -------------------------
print("""
Notes / next steps:
- If your dataset is heavily imbalanced, consider enabling SMOTE in the pipeline (see commented code).
- Increase n_iter in RandomizedSearchCV if you have more compute/time for better hyperparameter search.
- If you want explicit early stopping for other models:
  - Use MLPClassifier (has early_stopping=True) for a neural net with epochs.
  - For sklearn hist-gradient boosting: use early_stopping=True param.
- To tune VotingClassifier weights, you can run a light grid/random search over weight combinations.
- Make sure 'churn.csv' and 'Churn' column names match your dataset, and update categorical/numeric column lists if needed.
""")


Categorical columns: ['customerID', 'gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'TotalCharges']
Numeric columns: ['SeniorCitizen', 'tenure', 'MonthlyCharges']

--- Starting RandomizedSearchCV for LogisticRegression ---
Fitting 4 folds for each of 30 candidates, totalling 120 fits
RandomizedSearchCV done for LogisticRegression in 1862.3s
Best params: {'clf__C': np.float64(0.5741157902710026), 'clf__penalty': 'l2', 'clf__solver': 'saga'}
Best CV score: 0.8441844101721375

--- Starting RandomizedSearchCV for RandomForest ---
Fitting 4 folds for each of 40 candidates, totalling 160 fits
RandomizedSearchCV done for RandomForest in 240.4s
Best params: {'clf__bootstrap': True, 'clf__max_depth': 23, 'clf__min_samples_leaf': 1, 'clf__min_samples_split': 8, 'clf__n_estimators': 323}
Best CV score: 0.8354

ValueError: 
All the 200 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
200 fits failed with the following error:
Traceback (most recent call last):
  File "C:\Users\visha\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\visha\anaconda3\Lib\site-packages\sklearn\base.py", line 1389, in wrapper
    return fit_method(estimator, *args, **kwargs)
  File "C:\Users\visha\anaconda3\Lib\site-packages\sklearn\pipeline.py", line 662, in fit
    self._final_estimator.fit(Xt, y, **last_step_params["fit"])
    ~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\visha\anaconda3\Lib\site-packages\xgboost\core.py